# 01 - Codebase Analysis: What Exists Today

**Goal:** Dissect the current spine segmentation codebases — training and inference — to understand what exists, what's shared, and what's missing before proposing improvements.

---

## Two Codebases, One Pipeline

```
spine-segmentation-feature-training-scripts/     segmentation/
┌──────────────────────────────────┐             ┌───────────────────────────┐
│ TRAINING (v0.17-rc1)             │             │ INFERENCE (v0.16.0)       │
│                                  │             │                           │
│ • Train SwinUNETR models         │   weights   │ • Load trained models     │
│ • Data augmentation              │ ──────────→ │ • Run 2-stage pipeline    │
│ • Validation + metrics           │             │ • SegRefine post-process  │
│ • TensorBoard logging            │             │ • Docker + CI/CD          │
│ • Save best_metric_model.pth     │             │ • ACR push               │
│                                  │             │                           │
│ Shared: config, classes,         │             │ Shared: config, classes,  │
│   transforms, dataset, model     │             │   transforms, dataset     │
└──────────────────────────────────┘             └───────────────────────────┘
```

The model weights are the **only artifact** that flows from training to inference. Everything else is duplicated.

## Software Items Inventory

### Training Codebase

```
spine-segmentation-feature-training-scripts/
│
├── Entry Points
│   ├── src/training_semantic.py     Training loop for region model (2mm)
│   ├── src/training_instance.py     Training loop for instance model (1mm)
│   ├── src/testing_semantic.py      Evaluation for region model
│   ├── src/testing_instance.py      Evaluation for instance model
│   ├── main_pipeline.py             Inference pipeline (also in training repo)
│   └── docker_entrypoint.py         Docker CLI wrapper
│
├── Core Modules
│   ├── src/config.py                Config loading (3 functions: training, testing, inference)
│   ├── src/classes.py               Vertebra class definitions & label mappings
│   ├── src/dataset.py               DataLoader factories (Standard, Cache, SmartCache)
│   ├── src/inference_pipeline.py    Inference engine orchestration
│   ├── src/inferrer.py              Sequential patch-based inference
│   ├── src/display.py               TensorBoard & CLI visualization
│   ├── src/utils.py                 Path utilities, scan name extraction
│   └── src/exceptions.py            Custom PipelineError
│
├── Transform Pipeline
│   ├── src/transform/custom.py      Custom MONAI transforms (padding, label mapping, centroid crop)
│   ├── src/transform/utils.py       Transform utilities
│   ├── src/transform/composition/semantic.py    Region task transforms
│   └── src/transform/composition/instance.py    Instance task transforms
│
├── Configuration
│   ├── config/spine_semantic_new_v8.yaml         Region model config
│   └── config/spine_instance_new_v2.yaml         Instance model config
│
├── Artifacts
│   ├── weights/training_spine_semantic_new_v8/    Region model weights
│   └── weights/training_spine_instance_new_v2/    Instance model weights
│
└── Infrastructure
    ├── Dockerfile                   Multi-stage build (Ubuntu 22.04 + CUDA 11.8)
    ├── .gitlab-ci.yml              CI/CD pipeline
    ├── requirements.txt            24 Python packages
    └── pyproject.toml              Ruff linter config
```

### Inference Codebase

```
segmentation/
│
├── Entry Points
│   ├── main_pipeline.py             Inference pipeline orchestration
│   └── docker_entrypoint.py         Docker CLI wrapper
│
├── Core Modules
│   ├── src/config.py                Config loading (same structure as training)
│   ├── src/classes.py               Same vertebra class definitions
│   ├── src/dataset.py               Same DataLoader factories
│   ├── src/inference_engine_v016.py 2-stage inference engine (796 lines)
│   ├── src/inferrer.py              Sequential inference & post-processing
│   ├── src/display.py               Display utilities
│   ├── src/utils.py                 Utilities
│   └── src/main_settings.py         CLI argument parser
│
├── Transform Pipeline
│   ├── src/transform/custom.py      Same custom transforms
│   ├── src/transform/utils.py       Same utilities
│   ├── src/transform/composition/region.py      Region transforms
│   └── src/transform/composition/instance.py    Instance transforms
│
├── Configuration
│   ├── config/vertebrae_type_spine_2mm_v2_azure_gpu_1_lumbosacral_castellvi.yaml
│   └── config/spine_sequential_1mm_azure_v6.yaml
│
├── Artifacts
│   ├── weight/training_vertebrae_type_spine_2mm_v2_.../last_model.pth     (187 MB)
│   └── weight/training_spine_sequential_1mm_azure_v6/best_metric_model.pth (187 MB)
│
└── Infrastructure
    ├── Dockerfile                   Multi-stage + SegRefine from ACR
    ├── .gitlab-ci.yml              quality → build → scan → push → cleanup
    ├── requirements.txt            Same 24 packages
    └── pyproject.toml              Same Ruff config
```

## Training Pipeline Flow

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                         TRAINING PIPELINE                                   │
│                                                                             │
│  ┌──────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐  │
│  │  CONFIG   │    │    DATA      │    │   TRAINING   │    │   OUTPUT     │  │
│  │          │    │              │    │              │    │              │  │
│  │ YAML     │───→│ CSV splits   │───→│ SwinUNETR    │───→│ best_model   │  │
│  │ config   │    │ NIfTI files  │    │ DiceCE loss  │    │ .pth         │  │
│  │          │    │ SmartCache   │    │ Adam optim   │    │              │  │
│  │ Defines: │    │ Transforms:  │    │ 50k iters    │    │ TensorBoard  │  │
│  │ • model  │    │ • normalize  │    │              │    │ logs         │  │
│  │ • data   │    │ • resample   │    │ Validation:  │    │              │  │
│  │ • train  │    │ • augment    │    │ every 500-   │    │ Checkpoint:  │  │
│  │ • test   │    │ • crop patch │    │ 1000 steps   │    │ • model      │  │
│  │          │    │              │    │ sliding win  │    │ • optimizer  │  │
│  └──────────┘    └──────────────┘    └──────────────┘    │ • scheduler  │  │
│                                                          └──────────────┘  │
│                                                                             │
│  Run twice: once for region (2mm), once for instance (1mm)                  │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Key parameters (from configs)

| Parameter | Region (semantic_v8) | Instance (instance_v2) |
|-----------|---------------------|------------------------|
| Spacing | 2.0 mm | 1.0 mm |
| Input shape | 128×128×96 | 128×128×96 |
| Features | 24 channels | 24 channels |
| Batch size | 8 | 2 |
| Max iterations | 50,000 | 50,000 |
| Val interval | 500 | 1,000 |
| SmartCache size | 96 samples | 64 samples |
| Loss | DiceCE (no background) | DiceCE (with background) |
| Output classes | ~10 regions | ~20 vertebrae |

## Inference Pipeline Flow

```
┌─────────────────────────────────────────────────────────────────────────┐
│                       INFERENCE PIPELINE                                │
│                                                                         │
│  Input: CT scan (.mha / .nii.gz)                                        │
│    │                                                                    │
│    ▼                                                                    │
│  [001] Format conversion                                                │
│    │   .mha → .nii.gz (if needed)                                       │
│    ▼                                                                    │
│  [002] REGION MODEL (2mm)                         ~10-30s               │
│    │   ├── Preprocess: normalize → crop → resample 2mm                  │
│    │   ├── SwinUNETR (96×96×96, 5 output channels)                     │
│    │   ├── Sliding window: 1 patch, 50% overlap                        │
│    │   └── Output: region mask (cervical/thoracic/lumbar/sacrum)        │
│    ▼                                                                    │
│  [003] INSTANCE MODEL (1mm)                       ~15-40s               │
│    │   ├── Preprocess: normalize → crop → resample 1mm                  │
│    │   ├── SwinUNETR (128×128×96, 20 output channels)                  │
│    │   ├── Sliding window: 4 patches, 80% overlap                      │
│    │   └── Output: individual vertebra labels (C1-S1)                   │
│    ▼                                                                    │
│  [004] Multi-label aggregation                                          │
│    │   Combine region + instance predictions                            │
│    │   Handle overlaps (vertebra vs pelvis regions)                     │
│    ▼                                                                    │
│  [005] SegRefine (optional)                       ~5-15s                │
│    │   C++ post-processing: boundary refinement                         │
│    │   inner/outer margin: 2mm                                          │
│    ▼                                                                    │
│  [006] Format conversion                                                │
│    │   .nii.gz → .mha (if input was .mha)                              │
│    ▼                                                                    │
│  Output: segmentation mask + timing CSV            Total: ~30-80s       │
└─────────────────────────────────────────────────────────────────────────┘
```

## Code Duplication Map

These modules exist in **both** codebases with nearly identical code:

```
TRAINING REPO                          INFERENCE REPO
─────────────                          ──────────────
src/config.py          ←── same ──→    src/config.py
  manage_training_configurations()       manage_training_configurations()
  manage_testing_configurations()        manage_testing_configurations()
  manage_infengine_configurations()      manage_infengine_configurations()

src/classes.py         ←── same ──→    src/classes.py
  get_model_classes()                    get_model_classes()
  label mappings (v17, sequential...)    label mappings (lumbosacral...)

src/dataset.py         ←── same ──→    src/dataset.py
  get_data_from_split()                  get_data_from_split()
  dataset_builder()                      dataset_builder()

src/transform/         ←── same ──→    src/transform/
  custom.py (PadToMinimumSized...)       custom.py
  composition/semantic.py                composition/region.py  (renamed!)
  composition/instance.py                composition/instance.py

src/inferrer.py        ←── same ──→    src/inferrer.py
  sequential_inference_testing()         sequential_inference_testing()

requirements.txt       ←── same ──→    requirements.txt
  24 identical packages                  24 identical packages

Dockerfile             ←── same ──→    Dockerfile
  Multi-stage, Ubuntu 22.04              Multi-stage, Ubuntu 22.04
  CUDA 11.8, PyTorch 2.6.0              CUDA 11.8, PyTorch 2.6.0
```

### What this means

- **~70% of the code is duplicated** between repos
- Bug fixes need to be applied in both places
- Config format must stay in sync manually
- A label mapping change requires updating both repos
- Different naming (`semantic.py` vs `region.py`) creates confusion

## Gap Analysis: What's Missing

| Category | Current State | Impact |
|----------|--------------|--------|
| **Data versioning** | Data on NAS, no tracking | Can't reproduce training runs |
| **Experiment tracking** | TensorBoard only | Can't compare runs or find best config |
| **Model registry** | `.pth` files in folders | Can't trace which model is deployed |
| **Data lineage** | None | FDA 510(k) non-compliance |
| **Shared code** | Copy-pasted between repos | Bugs fixed in one, not the other |
| **Config versioning** | Filename-based (manual) | `v6`, `v8` — who decides? |
| **Weights management** | Bundled in Docker at build time | Can't swap model without rebuild |
| **Pipeline automation** | Manual scripts | No `dvc repro` equivalent |
| **Testing** | No unit tests | Changes can break silently |
| **Documentation** | READMEs only | No architectural docs |

### FDA 510(k) Risk Assessment

```
CRITICAL (blocks submission):
  ✗ No data lineage — can't prove which data trained which model
  ✗ No model change tracking — can't show model evolution
  ✗ No validation documentation — metrics exist but aren't structured

HIGH (needs fixing before audit):
  ✗ No reproducibility — can't regenerate exact training run
  ✗ No software versioning beyond git tags
  ✗ No SBOM for training environment (only inference has one)

MEDIUM (should improve):
  ✗ No unit tests for data transforms
  ✗ Duplicated code increases error surface
  ✗ Manual config management
```

## Config Deep Dive

The YAML configs are well-structured but have drift between repos:

### Training Configs (v0.17)

```yaml
# spine_semantic_new_v8.yaml
task:
  name: 'spine_semantic_new'          # naming convention: {body}_{type}_{modifier}
  version: 'v8'                        # manual version string
dataset:
  name: 'temp_data_transfer'           # ← temporary name, not descriptive
  split_name: '101225_s1ok'            # ← date-based, cryptic
model:
  input_shape: [128, 128, 96]
  features_chan: 24
  classes: 'v17'                       # ← class group name
training:
  max_iterations: 50000
  batch_size: 8
  optimizer: {learning_rate: 1.0e-3}
```

### Inference Configs (v0.16)

```yaml
# vertebrae_type_spine_2mm_v2_azure_gpu_1_lumbosacral_castellvi.yaml
task:
  name: 'vertebrae_type_spine_2mm'     # different naming convention!
  version: 'v2_azure_lumbosacral_castellvi'
dataset:
  name: 'temp_data_transfer'           # same name but possibly different data
model:
  input_shape: [96, 96, 96]            # different input shape!
  features_chan: 24
  classes: 'lumbosacral'               # different class group!
```

### Issues

| Issue | Example |
|-------|---------|
| **Naming inconsistency** | `spine_semantic_new` (training) vs `vertebrae_type_spine_2mm` (inference) |
| **Different input shapes** | 128×128×96 (training) vs 96×96×96 (inference) for region model |
| **Different class groups** | `v17` (training) vs `lumbosacral` (inference) |
| **Unclear dataset refs** | `temp_data_transfer` — what does this actually point to? |
| **Cryptic split names** | `101225_s1ok` — date + shorthand, not documented |
| **Version drift** | Training is 2 versions ahead of inference configs |

## Docker & CI/CD Architecture

```
CURRENT CI/CD (inference repo only)

Developer pushes git tag (v0.16.0)
    │
    ▼
GitLab CI triggers:
    │
    ├── [quality] ruff-lint, ruff-format, pip-audit, SBOM
    │
    ├── [build] docker build --platform linux/amd64
    │     ├── Stage 1 (builder): Ubuntu + CUDA + PyTorch + deps
    │     ├── Stage 2 (production): Copy packages + CUDA libs
    │     ├── COPY --from=maiamvpacr.azurecr.io/segrefine/base:latest
    │     ├── Pre-compile .py → .pyc (security)
    │     └── Bundle weights + configs inside image
    │
    ├── [scan] Trivy security scan (HIGH/CRITICAL)
    │
    ├── [push] docker push → maiamvpacr.azurecr.io/verifine-spine/segmentation/base:0.16.0
    │
    └── [cleanup] Remove local images

TRAINING REPO: Has CI/CD too, same structure.
But training is run manually, not via CI/CD pipeline.
```

### What's Good

- Multi-stage Docker build (smaller final image)
- Security scanning (Trivy + pip-audit)
- SBOM generation (CycloneDX)
- Pre-compiled bytecode (code obfuscation)
- Tag-based releases (not every commit)

### What's Missing

- No CI/CD for training pipeline (training is manual)
- Weights bundled in image → image is ~4GB, rebuild to swap model
- No automated testing (no test stage in CI)
- No model validation gate (model goes straight to image)
- Training repo CI builds a Docker image too, but it's not clear when/how it's used

## Dependency Analysis

Both repos use identical dependencies:

```
Core Stack:
├── PyTorch 2.6.0 + CUDA 11.8     (DL framework)
├── MONAI 1.4.0                    (medical imaging)
├── nibabel 5.0.0                  (NIfTI I/O)
├── SimpleITK 2.3.1               (image I/O + conversion)
├── numpy 1.26.4                   (arrays)
├── scipy 1.11.4                   (scientific computing)
├── scikit-image 0.22.0            (image processing)
├── scikit-learn 1.3.2             (DBSCAN clustering in inference)
├── pandas 2.1.4                   (CSV/DataFrame)
├── einops 0.7.0                   (tensor reshaping for Swin)
└── timm 0.9.12                    (vision model backbones)

Utilities:
├── PyYAML 6.0.1
├── tqdm 4.66.1
├── colorama 0.4.6
├── tabulate 0.9.0
└── pathlib2 2.3.7
```

This is good — it means a shared package would have zero dependency conflicts.

## Summary: Current State Assessment

```
STRENGTHS                              WEAKNESSES
─────────                              ──────────
✓ Config-driven pipeline               ✗ No data versioning (DVC)
✓ Well-structured YAML configs         ✗ No experiment tracking (MLflow)
✓ Multi-stage Docker builds            ✗ No model registry
✓ Security scanning (Trivy, SBOM)      ✗ ~70% code duplication
✓ Two-stage architecture (smart)       ✗ No shared code package
✓ SmartCache for memory efficiency     ✗ Config drift between repos
✓ Distributed training ready           ✗ No unit tests
✓ SegRefine integration                ✗ Weights bundled in Docker
                                       ✗ No FDA-ready lineage
                                       ✗ Training is fully manual
```

**The code quality is good. The architecture is good. What's missing is the infrastructure around it** — versioning, tracking, automation, and traceability.

**Next:** Target architecture — how to restructure this with MLOps.